# Modelflow parsing notebook

This notebook gives live, runnable examples for:

1. `model_parse()` in `modelpattern.py`
2. `BaseModel.analyzemodelnew()` in `modelclass.py`
3. Dependency graph construction and solution ordering

It stays close to the current Modelflow token structure:

```python
nterm = namedtuple('nterm', ['number', 'op', 'var', 'lag'])
```


## Self-contained demo

The notebook includes a small parser and analyzer that mirror the core behavior:

- formulas are found as `FRML ... $`
- expressions are tokenized into `nterm(number, op, var, lag)`
- `+0` and `-0` lags are normalized to `''`
- the analyzer identifies LHS variables, RHS dependencies, lags, and same-period graph edges


In [1]:
import re
from collections import namedtuple
from graphlib import TopologicalSorter, CycleError
from pprint import pprint

nterm = namedtuple("nterm", ["number", "op", "var", "lag"])
fatoms = namedtuple("fatoms", ["whole", "frml", "frmlname", "expression"])

funkname = "DLOG SUM_EXCEL DIFF MIN MAX FLOAT NORM.CDF NORM.PPF ABS MOVAVG PCT_GROWTH LOG EXP".split()
funkname2 = [f + r"(?=\()" for f in funkname]

namepat_ng = r'(?:[A-Za-z_{][A-Za-z_{}0-9]*)'
namepat = r'(' + namepat_ng + r')'
lagpat = r'(?:\(([+-][0-9]+)\))?'
numpat = r'((?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)'
opname = r'\*\* != >= <= == [=+\-/*@|()$><,.\]\[]'.split()
oppat = '(' + '|'.join(['(?:' + i + ')' for i in funkname2 + opname]) + ')'
udtrykpat = numpat + '|' + oppat + '|' + namepat + lagpat
expressionre = re.compile(udtrykpat)

frmlpat = r'(FRML [^$]*\$)'
ws = r'[\s]+'
ws2 = r'[\s]*'
upat = r'([^$]*\$)'
optionpat = r'(?:[<][^>]*[>])?'
splitpat = namepat + ws + '(' + namepat_ng + '?' + ws2 + optionpat + ')' + ws + upat

AEQUAL = nterm('', '=', '', '')


## Parser helpers


In [2]:
def find_frml(equations: str):
    return re.findall(frmlpat, equations, flags=re.IGNORECASE)

def split_frml(frml: str):
    m = re.search(splitpat, frml, flags=re.IGNORECASE)
    if m:
        return fatoms(m.group(0), m.group(1), m.group(2), m.group(3))
    raise ValueError(f"Could not split FRML: {frml}")

def model_parse_demo(equations: str):
    out = []
    for frml in find_frml(equations.upper()):
        f = split_frml(frml)
        tokens = []
        for t in expressionre.findall(f.expression):
            lag = '' if t[3] in ('+0', '-0') else t[3]
            tokens.append(nterm(t[0], t[1], t[2], lag))
        out.append((f, tokens))
    return out


## Example 1: tokenization


In [3]:
eq_text = '''
FRML <IDENT> A = B(-1) + 3.5 * MAX(C,D) $
FRML <IDENT> X = Y(+0) + Z(-0) + LOG(Q) $
'''

parsed = model_parse_demo(eq_text)

for f, tokens in parsed:
    print("\nFRML name/options:", f.frmlname)
    print("Expression:", f.expression.strip())
    print("Tokens:")
    for tok in tokens:
        print("   ", tok)



FRML name/options: <IDENT>
Expression: A = B(-1) + 3.5 * MAX(C,D) $
Tokens:
    nterm(number='', op='', var='A', lag='')
    nterm(number='', op='=', var='', lag='')
    nterm(number='', op='', var='B', lag='-1')
    nterm(number='', op='+', var='', lag='')
    nterm(number='3.5', op='', var='', lag='')
    nterm(number='', op='*', var='', lag='')
    nterm(number='', op='MAX', var='', lag='')
    nterm(number='', op='(', var='', lag='')
    nterm(number='', op='', var='C', lag='')
    nterm(number='', op=',', var='', lag='')
    nterm(number='', op='', var='D', lag='')
    nterm(number='', op=')', var='', lag='')
    nterm(number='', op='$', var='', lag='')

FRML name/options: <IDENT>
Expression: X = Y(+0) + Z(-0) + LOG(Q) $
Tokens:
    nterm(number='', op='', var='X', lag='')
    nterm(number='', op='=', var='', lag='')
    nterm(number='', op='', var='Y', lag='')
    nterm(number='', op='+', var='', lag='')
    nterm(number='', op='', var='Z', lag='')
    nterm(number='', op='+', v

`B(-1)` should appear as one token with `var='B'` and `lag='-1'`.
`MAX` and `LOG` should appear in `op`.
`Y(+0)` and `Z(-0)` should be normalized to empty lag.


## Analysis helpers


In [4]:
def lag_to_int(lag: str):
    return 0 if lag == '' else int(lag)

def analyze_equation_tokens(tokens):
    assigpos = tokens.index(AEQUAL)
    lhs = tokens[:assigpos]
    rhs = tokens[assigpos + 1:]

    lhs_vars = [t.var for t in lhs if t.var]
    endo = lhs_vars[-1]

    rhs_vars = [t.var for t in rhs if t.var]
    rhs_current = [t.var for t in rhs if t.var and t.lag == '']
    rhs_lags = [lag_to_int(t.lag) for t in rhs if t.var and t.lag != '']

    max_lag = max([-x for x in rhs_lags if x < 0], default=0)
    max_lead = max([x for x in rhs_lags if x > 0], default=0)

    return {
        "endogenous": endo,
        "lhs_vars": lhs_vars,
        "rhs_vars_all": rhs_vars,
        "rhs_current_vars": rhs_current,
        "max_lag": max_lag,
        "max_lead": max_lead,
        "tokens": tokens,
    }

def analyze_model_demo(equations: str):
    parsed = model_parse_demo(equations)
    analyzed = []
    for f, tokens in parsed:
        info = analyze_equation_tokens(tokens)
        info["frml"] = f
        analyzed.append(info)
    return analyzed


## Example 2: equation analysis


In [5]:
model_text = '''
FRML <BEHAVIORAL> C   = A*YD + B*C(-1) $
FRML <IDENTITY>   YD  = W + TR $
FRML <IDENTITY>   GDP = C + I + G $
'''

analysis = analyze_model_demo(model_text)

for eq in analysis:
    print("\n---")
    print("FRML:", eq["frml"].frmlname)
    print("Expression:", eq["frml"].expression.strip())
    print("Endogenous :", eq["endogenous"])
    print("RHS vars all:", eq["rhs_vars_all"])
    print("Current RHS :", eq["rhs_current_vars"])
    print("Max lag     :", eq["max_lag"])
    print("Max lead    :", eq["max_lead"])



---
FRML: <BEHAVIORAL>
Expression: C   = A*YD + B*C(-1) $
Endogenous : C
RHS vars all: ['A', 'YD', 'B', 'C']
Current RHS : ['A', 'YD', 'B']
Max lag     : 1
Max lead    : 0

---
FRML: <IDENTITY>
Expression: YD  = W + TR $
Endogenous : YD
RHS vars all: ['W', 'TR']
Current RHS : ['W', 'TR']
Max lag     : 0
Max lead    : 0

---
FRML: <IDENTITY>
Expression: GDP = C + I + G $
Endogenous : GDP
RHS vars all: ['C', 'I', 'G']
Current RHS : ['C', 'I', 'G']
Max lag     : 0
Max lead    : 0


## Dependency graph and ordering


In [6]:
def build_dependency_graph(analysis):
    endo_set = {eq["endogenous"] for eq in analysis}
    graph = {}
    for eq in analysis:
        endo = eq["endogenous"]
        deps = {v for v in eq["rhs_current_vars"] if v in endo_set and v != endo}
        graph[endo] = deps
    return graph

def topo_or_cycle(graph):
    try:
        order = list(TopologicalSorter(graph).static_order())
        return {"kind": "topological_order", "value": order}
    except CycleError as e:
        return {"kind": "cycle", "value": e.args}

graph = build_dependency_graph(analysis)
print("Dependency graph:")
pprint(graph)

print("\nOrdering result:")
pprint(topo_or_cycle(graph))


Dependency graph:
{'C': {'YD'}, 'GDP': {'C'}, 'YD': set()}

Ordering result:
{'kind': 'topological_order', 'value': ['YD', 'C', 'GDP']}


In this toy model, a valid order is roughly `YD -> C -> GDP`.
Lagged terms such as `C(-1)` do not create same-period simultaneity.


## Example 3: simultaneous block


In [7]:
simul_text = '''
FRML <TEST> A = B + 1 $
FRML <TEST> B = A + 2 $
'''

sim_analysis = analyze_model_demo(simul_text)
sim_graph = build_dependency_graph(sim_analysis)

print("Dependency graph:")
pprint(sim_graph)

print("\nOrdering result:")
pprint(topo_or_cycle(sim_graph))


Dependency graph:
{'A': {'B'}, 'B': {'A'}}

Ordering result:
{'kind': 'cycle', 'value': ('nodes are in a cycle', ['A', 'B', 'A'])}


This cycle is the signature of a simultaneous block.


## Optional: use the real Modelflow modules

If the repository is available locally, adapt and run the next cell.


In [8]:
# import sys
# repo_root = r"/path/to/Modelflow2"
# if repo_root not in sys.path:
#     sys.path.insert(0, repo_root)
#
# from modelflow.modelpattern import model_parse, udtryk_parse, nterm
#
# model = 'FRML <TEST> A = B(-1) + 3.5 * MAX(C,D) $'
# real = model_parse(model)
# for f, toks in real:
#     print(f)
#     for t in toks:
#         print(t)
